# Train YOLO26x — Transformer Parts (transformer, wire)
Runtime → Change runtime type → **GPU** before running.

Trains from the consolidated dataset built by `scripts/build_dataset.py` (4 annotators merged, labels remapped to `transformer`/`wire`, images already CLAHE-preprocessed — **do not** preprocess again here).

In [ ]:
# YOLO26 needs a recent ultralytics; latest pip has it.
!pip install -q -U ultralytics

Upload `YOLO_thermal.zip` — zip the **contents** of the `YOLO_thermal/` folder so `train/` and `valid/` (each with `images/` + `labels/`) sit at the zip root.

In [ ]:
import zipfile, os
from google.colab import files
uploaded = files.upload()  # pick your dataset.zip
zip_name = next(iter(uploaded))
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/dataset")
print("extracted:", os.listdir("/content/dataset"))

In [ ]:
# Write the dataset config. names order MUST match labelImg classes.txt.
data_yaml = "/content/dataset/data.yaml"
with open(data_yaml, "w") as f:
    f.write(
        "path: /content/dataset\n"
        "train: train/images\n"
        "val: valid/images\n"
        "nc: 2\n"
        "names: ['transformer', 'wire']\n"
    )
print(open(data_yaml).read())

In [ ]:
from ultralytics import YOLO
# YOLO26x: largest, NMS-free end-to-end detector. Heavy for ~1.5k images,
# so we lean on augmentation + early stopping to avoid overfitting.
model = YOLO("yolo26x.pt")
model.train(
    data=data_yaml,
    epochs=150, imgsz=640, patience=30,
    batch=8,            # x is heavy: drop to 4 if a T4 OOMs, or batch=-1 to auto-fit
    # Thermal-tuned aug: NO hue shift (it would scramble the heat palette);
    # geometric aug + mosaic help the minority 'transformer' class generalize.
    hsv_h=0.0, hsv_s=0.2, hsv_v=0.2,
    fliplr=0.5, flipud=0.0, degrees=5.0,
    mosaic=1.0, close_mosaic=10,
)

In [ ]:
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50:   ", metrics.box.map50)

In [ ]:
# Optional: eyeball predictions on the validation images (saved under runs/detect/predict).
model.predict("/content/dataset/valid/images", conf=0.25, save=True)

In [ ]:
# Download the trained weights to your machine.
from google.colab import files
files.download("runs/detect/train/weights/best.pt")

Put the downloaded `best.pt` into your project's `models/` folder, then run the API: `THERMAL_WEIGHTS=models/best.pt uvicorn api:app --reload`.